In [6]:
import os
import argparse
import numpy as np
import torch
import qutip as qt
# Import definitions directly from your training script file (e.g., train.py)
# Make sure SingleQubitGateControlEnv and ContinuousPyTorchPolicy are accessible
from RL1Q import SingleQubitGateControlEnv, ContinuousPyTorchPolicy, HS

def test_checkpoint(checkpoint_path: str, num_eval_episodes: int = 100):
    device = torch.device("cpu")

    # 1. Check file existence
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint file not found at: '{checkpoint_path}'")

    print(f"[Loading] Checkpoint: {checkpoint_path}")

    # 2. Instantiate Environment & Policy Network
    env = SingleQubitGateControlEnv(n_segments=8, pulse_duration=20.0, n_gate_reps=1)
    
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    
    policy = ContinuousPyTorchPolicy(state_dim=state_dim, action_dim=action_dim, hidden_size=HS).to(device)

    # Then load as usual:
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        policy.load_state_dict(checkpoint['model_state_dict'])
        saved_iter = checkpoint.get('iteration', 'N/A')
        saved_fid = checkpoint.get('best_mean_fidelity', 'N/A')
        print(f"[Metadata] Trained Iteration: {saved_iter} | Saved Best Mean Fidelity: {saved_fid}")
    else:
        policy.load_state_dict(checkpoint)

    # Set to evaluation mode
    policy.eval()

    # 4. Run Deterministic/Greedy Evaluation
    print(f"\n[Evaluating] Running {num_eval_episodes} random Haar-state evaluation episodes...")
    
    fidelities = []
    actions_recorded = []

    with torch.no_grad():
        for ep in range(num_eval_episodes):
            state, _ = env.reset()
            done = False
            ep_actions = []

            while not done:
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                
                # Deterministic mean action forward pass
                action_mean = policy(state_t).squeeze(0).numpy()
                
                state, reward, done, _, info = env.step(action_mean)
                ep_actions.append(action_mean)

            fidelities.append(info['base_fidelity'])
            actions_recorded.append(ep_actions)

    # 5. Metrics & Summary Report
    fidelities = np.array(fidelities)
    mean_fid = np.mean(fidelities)
    std_fid = np.std(fidelities)
    min_fid = np.min(fidelities)
    max_fid = np.max(fidelities)

    print("\n" + "=" * 50)
    print(" 📊 EVALUATION SUMMARY REPORT")
    print("=" * 50)
    print(f" Total Evaluation Runs : {num_eval_episodes}")
    print(f" Mean Gate Fidelity   : {mean_fid:.6f} ± {std_fid:.6f}")
    print(f" Min Gate Fidelity    : {min_fid:.6f}")
    print(f" Max Gate Fidelity    : {max_fid:.6f}")
    print(f" Infidelity (1 - F)   : {1.0 - mean_fid:.6e}")
    print("=" * 50)

    # Return sample action pulse trajectory from the last episode
    sample_pulse = np.array(actions_recorded[-1]) # shape: (n_segments, 2)
    return fidelities, sample_pulse


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Test and Evaluate Trained Gate Policy Checkpoint.")
    parser.add_argument("--ckpt", type=str, default="./ckpt_1q/best_policy.pt", help="Path to .pt checkpoint file")
    parser.add_argument("--episodes", type=int, default=100, help="Number of test evaluation episodes")
    
    #args = parser.parse_args()
    args = parser.parse_args([])
    
    test_checkpoint(args.ckpt, args.episodes)

[Loading] Checkpoint: ./ckpt_1q/best_policy.pt
[Metadata] Trained Iteration: 968 | Saved Best Mean Fidelity: 0.9985489454004766

[Evaluating] Running 100 random Haar-state evaluation episodes...

 📊 EVALUATION SUMMARY REPORT
 Total Evaluation Runs : 100
 Mean Gate Fidelity   : 0.999418 ± 0.000260
 Min Gate Fidelity    : 0.998702
 Max Gate Fidelity    : 0.999863
 Infidelity (1 - F)   : 5.820424e-04
